Solution of Navier Stokes equations

## Install Fenics and other dependencies

In [1]:
# Setup system environment and define utility functions and variables
from google.colab import files
try:
    from dolfin import *; from mshr import *
except ImportError as e:
    !apt-get install -y -qq software-properties-common python-software-properties module-init-tools
    !add-apt-repository -y ppa:fenics-packages/fenics
    !apt-get update -qq
    !sed -e 's:artful:bionic:' /etc/apt/sources.list.d/fenics-packages-ubuntu-fenics-artful.list > temp
    !mv temp /etc/apt/sources.list.d/fenics-packages-ubuntu-fenics-artful.list
    !sed -e 's:artful:bionic:' /etc/apt/sources.list > temp
    !mv temp /etc/apt/sources.list
    !apt-get update -qq
    !apt install -y --no-install-recommends fenics    
    from dolfin import *; from mshr import *
    
import matplotlib.pyplot as plt
from IPython.display import clear_output, display, update_display
import time
import dolfin.common.plotting as fenicsplot
import os, sys, shutil

dolfin_version = dolfin.__version__
print ('dolfin version:', dolfin_version)

E: Package 'python-software-properties' has no installation candidate
Get:1 http://ppa.launchpad.net/fenics-packages/fenics/ubuntu bionic InRelease [15.4 kB]
Hit:2 http://archive.ubuntu.com/ubuntu bionic InRelease
Get:3 http://archive.ubuntu.com/ubuntu bionic-updates InRelease [88.7 kB]
Get:4 http://ppa.launchpad.net/graphics-drivers/ppa/ubuntu bionic InRelease [21.3 kB]
Get:5 http://archive.ubuntu.com/ubuntu bionic-backports InRelease [74.6 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu bionic-cran35/ InRelease [3,626 B]
Get:7 http://ppa.launchpad.net/marutter/c2d4u3.5/ubuntu bionic InRelease [15.4 kB]
Get:8 http://security.ubuntu.com/ubuntu bionic-security InRelease [88.7 kB]
Ign:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  InRelease
Get:10 http://ppa.launchpad.net/fenics-packages/fenics/ubuntu bionic/main amd64 Packages [5,700 B]
Ign:11 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  InRelease
Get:12 ht

In [0]:
def plot_compact(u, t, stepcounter, QQ): # Compact plot utility function
    if stepcounter % 5 == 0:
        uEuclidnorm = project(sqrt(inner(u, u)), QQ); ax.cla(); fig = plt.gcf(); fig.set_size_inches(16, 4)
        plt.subplot(1, 2, 1); pp = plot(uEuclidnorm); plt.title("Velocity at t=%f" % (t)) # Plot norm of velocity
        if t == 0.: plt.axis(G); plt.colorbar(pp, shrink=0.5); 
        plt.subplot(1, 2, 2);
        if t == 0.: plot(Q.mesh()); plt.title("Mesh") # Plot mesh
        plt.tight_layout(); dpl = display(pl, display_id="test");

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/drive


In [4]:
!ls "/content/drive/My Drive/NSE/NS_3Dsphere/mesh"

3Dsphere.geo	  3Dsphere.msh	 boundaries.xdmf  domain.xdmf
3Dsphere.geo.opt  boundaries.h5  domain.h5	  mesh-convert.py


## 3D Sphere

In [0]:
#!/usr/bin/env python
from fenics import *
import numpy as np
import time

eps = 1e-5

def readMesh(path):
    mesh = Mesh()
    with XDMFFile(path + "domain.xdmf") as infile:
        infile.read(mesh)
    mvc = MeshValueCollection("size_t", mesh, 2)
    with XDMFFile(path + "boundaries.xdmf") as infile:
        infile.read(mvc, "boundaries")
    boundaries = cpp.mesh.MeshFunctionSizet(mesh, mvc)
    return (mesh, boundaries)

In [0]:
def boundaryCondition(mesh, boundaries, W):
    #v_in = Expression(("4 * 0.3 * (x[1] * (0.41 - x[1])) / (0.41 * 0.41)", "0."), degree=2)
    v_in = Expression(("4 * 0.3 * (x[1] * (0.41 - x[1])) * (x[2] * (0.41 - x[2])) / (0.41 * 0.41)", "0.", "0."), degree = 3)
    p_out = Constant(0.0)
    zero_vec = Constant((0., 0., 0.))

    bcp_inflow = DirichletBC(W.sub(0), v_in, boundaries, 4)
    bcp_outflow = DirichletBC(W.sub(1), p_out, boundaries, 6)
    noslip_boundaries = DirichletBC(W.sub(0), zero_vec, boundaries, 5)
    sphere_boundaries = DirichletBC(W.sub(0), zero_vec, boundaries, 7)

    bcs = [bcp_inflow, bcp_outflow, noslip_boundaries, sphere_boundaries]

    return bcs

In [0]:
def force(mesh, boundaries, w, nu):
    #Initialize mesh function for boundary domains
    ds_sphere = Measure('ds', domain = mesh, subdomain_data = boundaries, subdomain_id = 7)

    #Calculate force
    n = FacetNormal(mesh)

    u = w.split()[0]
    p = w.split()[1]

    F = nu*dot(grad(u), n) - p*n

    D = -F[0] * ds_sphere
    L = -F[1] * ds_sphere

    drag_force = assemble(D)
    lift_force = assemble(L)

    Umean = 0.2
    L = 0.1
    drag_coeff = drag_force * 2 / (L * Umean**2)
    lift_coeff = lift_force * 2 / (L * Umean**2)

    return (drag_force, lift_force, drag_coeff, lift_coeff)

In [0]:
def ufl(mesh, f, dt, nu, theta):
    Ve = VectorElement("CG", mesh.ufl_cell(), 2)
    Qe = FiniteElement("CG", mesh.ufl_cell(), 1)
    W = FunctionSpace(mesh, Ve * Qe)

    # Define test functions
    w = Function(W)
    u, p = split(w)
    (v, q) = TestFunctions(W)
    w0 = Function(W)
    u0, _ = split(w0)

    # Define coefficients
    k = dt
    um = theta*u + (1.0-theta)*u0

    F = (1/k)*inner(u - u0, v)*dx + inner(grad(um)*um, v)*dx \
        + nu*inner(grad(um), grad(v))*dx + inner(grad(p), v)*dx + div(um)*q*dx\
        - inner(f, v)*dx

    #Alternative formulations    
    #F = (1/k)*inner(u - u0, v)*dx + inner(grad(um)*um, v)*dx \
    #    + nu*inner(grad(um), grad(v))*dx - inner(div(v),p)*dx + inner(div(u),q)*dx - inner(f, v)*dx
    
    #F = (1/k)*inner(u - u0, v)*dx\
    #    + theta*nu*inner(grad(u), grad(v))*dx + (1-theta)*nu*inner(grad(u), grad(v))*dx\
    #    + theta*inner(grad(u)*u, v)*dx + (1-theta)*inner(grad(u0)*u0, v)*dx\
    #    - inner(div(v),p)*dx - inner(div(u),q)*dx - inner(f, v)*dx

    #F = (1/k)*inner(u - u0, v)*dx + inner(grad(um)*um, v)*dx \
    #    +2*nu*inner(epsilon(um),epsilon(v))*dx - inner(p,div(v))*dx + div(um)*q*dx\
    #    - inner(f, v)*dx
        
    # stabilization parameter
    h = CellDiameter(mesh)
    u_norm = sqrt(dot(w.split()[0], w.split()[0]))
    tau = ((2.0/k)**2 + (2.0*u_norm/h)**2 + (4.0*nu/h**2)**2)**(-0.5)
    F += tau*(inner(grad(p) + grad(um)*um, grad(q) + grad(um)*v) + inner(div(um), div(v)))*dx

    J = derivative(F, w)

    return (F, J, W, w, w0)

In [0]:
def main():
    meshPath = "/content/drive/My Drive/NSE/NS_3Dsphere/mesh/"

    dt = 0.1
    f = Constant((0, 0, 0))
    mu = 0.001
    rho = 1
    nu = mu/rho
    theta = 0.5  # Crank-Nicholson
    T = 1.0

    (mesh, boundaries) = readMesh(meshPath)
    (F, J, W, w, w0) = ufl(mesh, f, dt, nu, theta)
    bcs = boundaryCondition(mesh, boundaries, W)

    problem = NonlinearVariationalProblem(F, w, bcs, J)
    solver = NonlinearVariationalSolver(problem)

    file = File("/content/drive/My Drive/NSE/NS_3Dsphere/output_sphere/velocity.pvd")

    values = []
    t = 0
    while t <= T:
        w0.assign(w)
        solver.solve()
        drag_force_t = force(mesh, boundaries, w, nu)[0]
        lift_force_t = force(mesh, boundaries, w, nu)[1]
        drag_coeff_t = force(mesh, boundaries, w, nu)[2]
        lift_coeff_t = force(mesh, boundaries, w, nu)[3]
        values.append(np.array([t,drag_force_t,lift_force_t, drag_coeff_t, lift_coeff_t]))
        file << w0.split()[0]
        info("Current time: {0:1.3f}".format(t))
        t += dt
    
    np.savetxt("/content/drive/My Drive/NSE/NS_3Dsphere/output_sphere/drag_lift.txt", values, delimiter=",")

    Re = 0.2*0.1/(mu/rho)

    print('Reynolds number: ', Re)

    print ('drag force: ', force(mesh, boundaries, w, nu)[0])
    print ('lift force: ', force(mesh, boundaries, w, nu)[1])
    print ('drag coeff: ', force(mesh, boundaries, w, nu)[2])
    print ('lift coeff: ', force(mesh, boundaries, w, nu)[3])

In [0]:
if __name__ == "__main__":

    main()

Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
           Consider using the option 'quadrature_degree' to reduce the number of points
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
           Consider using the option 'quadrature_degree' to reduce the number of points


## Plot results

In [0]:
import pandas as pd

data = pd.read_csv('/content/drive/My Drive/NSE/NS_3Dsphere/output_sphere/drag_lift.txt', header = None)
data.rename(columns={0:'t', 
                     1:'drag_force',
                     2:'lift_force',
                     3:'drag_coeff',
                     4:'lift_coeff'}, 
                 inplace=True)
data.head()

In [0]:
#import matplotlib as plt
import matplotlib.pyplot as plt

#pd.DataFrame.plot(x = 't', y = 'drag_force', data = data)
plt.figure(figsize=(10, 10))
plt.scatter(data['t'][10:], data['drag_force'][10:])
plt.show()

In [0]:
#import matplotlib as plt
import matplotlib.pyplot as plt

#pd.DataFrame.plot(x = 't', y = 'drag_force', data = data)
plt.figure(figsize=(10, 10))
plt.scatter(data['t'][2:], data['lift_force'][2:])
plt.show()

In [0]:
#import matplotlib as plt
import matplotlib.pyplot as plt

#pd.DataFrame.plot(x = 't', y = 'drag_force', data = data)
plt.figure(figsize=(10, 10))
plt.scatter(data['t'][10:], data['drag_coeff'][10:])
plt.show()

In [0]:
#import matplotlib as plt
import matplotlib.pyplot as plt

#pd.DataFrame.plot(x = 't', y = 'drag_force', data = data)
plt.figure(figsize=(10, 10))
plt.scatter(data['t'][2:], data['lift_coeff'][2:])
plt.show()